In [0]:
from pyspark.sql.functions import to_date,sum,round,count,avg

In [0]:
daily_revenue = spark.table("nyc_taxi.lakehouse.silver_taxi")\
    .groupBy(to_date("pickup_datetime").alias("trip_date")).agg(
        round(sum("total_amount"),2).alias("daily_revenue")
    )


In [0]:
daily_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "nyc_taxi.lakehouse.gold_daily_revenue"
    )

In [0]:
peak_hour = spark.table("nyc_taxi.lakehouse.silver_taxi")\
    .groupby("pickup_hour").agg(count("*").alias("total_trips"))\
    .orderBy("total_trips",descending=True)

peak_hour.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("nyc_taxi.lakehouse.gold_peak_hour")

In [0]:
top_pickup_locations = (spark.table("nyc_taxi.lakehouse.silver_taxi")\
    .groupBy("PULocationID").agg(count("*").alias("trip_count"))\
    .orderBy("trip_count",ascending=False))

top_pickup_locations.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("nyc_taxi.lakehouse.gold_top_pickup_locations")

In [0]:
payment_analysis = (
    spark.table("nyc_taxi.lakehouse.silver_taxi")
    .groupBy("payment_type")
    .agg(
        count("*").alias("total_trips"),
        avg("total_amount").alias("avg_trip_amount")
    )
)

payment_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "nyc_taxi.lakehouse.gold_payment_analysis"
    )